# Aşama 1 — Tekrar (5 başlangıç) ve kontrol bitkisi testi (HuBERT-ECG, 6 saatlik dilim)

**1)** **Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU → Kaydet**

**2)** **Çalışma zamanı → Tümünü çalıştır** (uyarı çıkarsa *Yine de çalıştır*), Drive iznini ver.

| Adım | İş | Süre (T4, tahmini) |
|---|---|---|
| 4 | Donmuş model: EKG + 5 farklı rastgele başlangıç; stres testi ve kontrol bitkisi testi | ~5 dk |
| 5 | İnce ayar: (EKG, rastgele) × 5 başlangıç × 12 bitki | ~30–40 dk (ilk bitkiden sonra gerçek süre yazılır) |
| 6 | Özet | < 1 dk |

Veri yüklemen gerekmez (GitHub'dan gelir). Her sonuç Drive'a (`bitkiEkg_sonuclar/asama1/`) yazılır;
oturum koparsa **Tümünü çalıştır** yeter, biten kısımlar atlanır. Ayarlar önceki deneylerle aynıdır; değiştirme.

## 1. GPU kontrolü

In [ ]:
import torch
assert torch.cuda.is_available(), 'GPU yok! Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU, sonra Tümünü çalıştır.'
print('GPU:', torch.cuda.get_device_name(0), '| torch', torch.__version__)

## 2. Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
KOK = Path('/content/drive/MyDrive/bitkiEkg_sonuclar/asama1')
KOK.mkdir(parents=True, exist_ok=True)
print('Sonuç klasörü:', KOK)

## 3. Kod ve veri

In [ ]:
!rm -rf /content/bitkiEkg
!git clone -q -b onDeneme --depth 1 https://github.com/melihakcam/bitkiEkg.git /content/bitkiEkg
!pip install -q "transformers>=4.40,<5" safetensors huggingface_hub
import sys
sys.path.insert(0, '/content/bitkiEkg/src')
import transformers
assert int(transformers.__version__.split('.')[0]) < 5, 'transformers 5 yüklü: Çalışma zamanı → Yeniden başlat, sonra Tümünü çalıştır.'
import json, time
from dataclasses import replace
import numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from bitki_ekg.dapt import hubert_govde
from bitki_ekg.ince_ayar import AYARLAR, lopo_calistir
from bitki_ekg.sonda import hubert_gomme

z = np.load('/content/bitkiEkg/data/colab/domates_ikili_6h_500.npz')       # 12 sulama bitkisi
X = np.tile(z['L'].astype(np.float32), (1, 12)); y, g = z['y'].astype(int), z['bitki'].astype(int)
k = np.load('/content/bitkiEkg/data/colab/domates_kontrol_6h_500.npz')     # 4 kontrol bitkisi
C = np.tile(k['L'].astype(np.float32), (1, 12)); yc, gc = k['y'].astype(int), k['bitki'].astype(int)
assert not set(np.unique(g)) & set(np.unique(gc))
print('Sulama:', X.shape, np.bincount(y), '| Kontrol:', C.shape, np.bincount(yc))
TEKRARLAR = [1, 2, 3, 4, 5]

## 4. Donmuş model: EKG ağırlıkları + 5 farklı rastgele başlangıç

- **stres:** 12 sulama bitkisinde LOPO (önceki sonuç: EKG 77,9 · rastgele 72,8).
- **kontrol:** 4 kontrol bitkisinde ilk/son gün ayrımı, LOPO (≈ %50 olmalı).
- **aktarım:** 12 sulama bitkisinde eğitilen stres modeli kontrol bitkilerine uygulanır; AUC ≈ 0,5 olmalı
  (yüksekse model zamanı/elektrot kaymasını öğreniyor).

In [ ]:
def sonda():
    return make_pipeline(StandardScaler(), LogisticRegression(C=1.0, max_iter=5000, random_state=42))

def lopo(E, yy, gg, test_ad):
    satir = []
    for b in np.unique(gg):
        tr, te = gg != b, gg == b
        m = sonda().fit(E[tr], yy[tr]); s = m.predict_proba(E[te])[:, 1]
        satir.append({'test': test_ad, 'bitki': int(b), 'dogruluk': accuracy_score(yy[te], s >= 0.5),
                      'auc': roc_auc_score(yy[te], s)})
    return satir

DONMUS = KOK / 'donmus_6h.csv'
if DONMUS.exists():
    print('Önceden tamamlanmış, atlandı')
else:
    satir = []
    for kol, onceden, tohum in [('ekg', True, 42)] + [(f'rastgele_t{t}', False, t) for t in TEKRARLAR]:
        t0 = time.time()
        govde = hubert_govde(onceden, tohum)
        E, Ec = hubert_gomme(govde, X, 'cuda'), hubert_gomme(govde, C, 'cuda')
        np.savez_compressed(KOK / f'gomme_{kol}.npz', E=E, Ec=Ec)
        del govde; torch.cuda.empty_cache()
        r = lopo(E, y, g, 'stres') + lopo(Ec, yc, gc, 'kontrol')
        s = sonda().fit(E, y).predict_proba(Ec)[:, 1]
        for b in np.unique(gc):
            m = gc == b
            r.append({'test': 'aktarim', 'bitki': int(b), 'dogruluk': accuracy_score(yc[m], s[m] >= 0.5),
                      'auc': roc_auc_score(yc[m], s[m])})
        satir += [dict(x, kol=kol) for x in r]
        print(f'{kol}: {time.time() - t0:.0f} s', flush=True)
    pd.DataFrame(satir).to_csv(DONMUS, index=False)
d = pd.read_csv(DONMUS)
print((d.groupby(['test', 'kol'])[['dogruluk', 'auc']].mean() * 100).round(1).unstack(0))

## 5. İnce ayar: (EKG, rastgele) × 5 başlangıç × 12 bitki

In [ ]:
toplam = 2 * len(TEKRARLAR); i = 0; t_bas = time.time()
for t in TEKRARLAR:
    for ad, onceden in (('ekg', True), ('rastgele', False)):
        i += 1
        print(f'=== [{i}/{toplam}] İnce ayar: {ad}, başlangıç {t} ===', flush=True)
        t0 = time.time()
        lopo_calistir(X, y, g, onceden, KOK / 'ince_ayar' / f'{ad}_t{t}', 'cuda', ayar=replace(AYARLAR, tohum=t))
        torch.cuda.empty_cache()
        gecen = time.time() - t_bas
        print(f'  bu kol {time.time() - t0:.0f} s | toplam {gecen / 60:.1f} dk | '
              f'kalan tahmini {gecen / i * (toplam - i) / 60:.0f} dk', flush=True)

## 6. Özet (bitince bana "bitti" yazman yeterli)

In [ ]:
satir = []
for f in sorted((KOK / 'ince_ayar').glob('*_t*/bitki_*.json')):
    r = json.loads(f.read_text(encoding='utf-8'))
    ad, t = f.parent.name.rsplit('_t', 1)
    satir.append({'kol': ad, 'tekrar': int(t), 'bitki': r['test_bitkisi'], 'dogruluk': r['dogruluk'],
                  'f1': r['f1'], 'auc': r['auc'], 'en_iyi_epoch': r['en_iyi_epoch']})
df = pd.DataFrame(satir)
df.to_csv(KOK / 'ince_ayar_tekrar_6h.csv', index=False)
print('Tamamlanan bitki sayısı:', df.groupby(['kol', 'tekrar']).size().to_dict())
t = df.groupby(['kol', 'tekrar'])[['dogruluk', 'auc']].mean().mul(100).round(1)
t['tek_sinif_bitki'] = df.assign(c=df.f1 < 0.05).groupby(['kol', 'tekrar']).c.sum()
print(t)
print((df.groupby('kol')[['dogruluk', 'auc']].mean() * 100).round(1))
p = df.pivot_table(index=['tekrar', 'bitki'], columns='kol', values='dogruluk')
print('EKG − rastgele (doğruluk, puan), tekrar başına:', ((p.ekg - p.rastgele).groupby('tekrar').mean() * 100).round(1).to_dict())
print('\nDonmuş model:')
d = pd.read_csv(KOK / 'donmus_6h.csv')
print((d.groupby(['test', 'kol'])[['dogruluk', 'auc']].mean() * 100).round(1).unstack(0))